## MODEL OPTIMIZATION

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_DIR = Path("..")

TRAIN_DIR = PROJECT_DIR / "dataset" / "Training"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

In [3]:
class_names = sorted(folder.name for folder in TRAIN_DIR.iterdir()if folder.is_dir())
class_to_label = {name: i for i, name in enumerate(class_names)}

paths, labels = [], []

for class_name in class_names:
    for path in (TRAIN_DIR / class_name).rglob("*"):
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            paths.append(str(path))
            labels.append(class_to_label[class_name])

X_train, X_val, y_train, y_val = train_test_split(np.array(paths),np.array(labels),test_size=0.20,random_state=42,stratify=labels)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))

Training images: 4480
Validation images: 1120


In [4]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(image,channels=1,expand_animations=False)
    image.set_shape([None, None, 1])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.ensure_shape(image, [224, 224, 1])

    return image, label

In [5]:
def create_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(preprocess_image,num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        dataset = dataset.shuffle(buffer_size=len(paths),seed=42)

    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = create_dataset(X_train, y_train, training=True)
val_dataset = create_dataset(X_val, y_val)

## 1 Hyperparameter tuning

In [6]:
def build_cnn(filters1, filters2, dense_units):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 1)),
        layers.Conv2D(filters1, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(filters2, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(dense_units, activation="relu"),
        layers.Dense(4, activation="softmax")])

    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

    return model

In [7]:
experiments = {"CNN_small": (16, 32, 64),
               "CNN_baseline": (32, 64, 128),
               "CNN_large": (64, 128, 128)}

In [8]:
results = []
for name, params in experiments.items():
    print("\nTraining:", name)
    tf.keras.backend.clear_session()
    model = build_cnn(*params)
    history = model.fit(train_dataset,validation_data=val_dataset,epochs=10,verbose=1)

    results.append({"Model": name,"Best Validation Accuracy": max(history.history["val_accuracy"]),"Best Validation Loss": min(history.history["val_loss"])})


Training: CNN_small

Epoch 1/10


d:\MRI-Brain-Tumor-Classification\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


280/280 ━━━━━━━━━━━━━━━━━━━━ 88s 298ms/step - accuracy: 0.7435 - loss: 0.6468 - val_accuracy: 0.8420 - val_loss: 0.4498
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 129s 238ms/step - accuracy: 0.8853 - loss: 0.2889 - val_accuracy: 0.8821 - val_loss: 0.3183
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 38s 131ms/step - accuracy: 0.9493 - loss: 0.1344 - val_accuracy: 0.8911 - val_loss: 0.2989
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 38s 133ms/step - accuracy: 0.9806 - loss: 0.0593 - val_accuracy: 0.8982 - val_loss: 0.3603
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 48s 165ms/step - accuracy: 0.9906 - loss: 0.0325 - val_accuracy: 0.9214 - val_loss: 0.3200
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 42s 147ms/step - accuracy: 0.9884 - loss: 0.0348 - val_accuracy: 0.9018 - val_loss: 0.3550
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 43s 149ms/step - accuracy: 0.9953 - loss: 0.0163 - val_accuracy: 0.9134 - val_loss: 0.3751
Epoch 8/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 89s 312ms/step - accuracy: 0.9975 - loss: 0.0084 - va

In [9]:
results_df = pd.DataFrame(results)
print(results_df.sort_values(by="Best Validation Accuracy",ascending=False))

          Model  Best Validation Accuracy  Best Validation Loss
1  CNN_baseline                  0.926786              0.267405
2     CNN_large                  0.926786              0.277657
0     CNN_small                  0.921429              0.298905


## 2 Learning-rate experiments